# Derin Öğrenme Tabanlı Modern Dil Modelleri

## Table of Content:
- [1. Recurrent Neural Networks(RNN):](#1.-Recurrent-Neural-Networks-(RNN):)

# 1. Recurrent Neural Networks (RNN):

- Sekans verileri işlemek için özel olarak tasarlanmış sinir ağlarıdır.
- Her zaman adımında, önceki zaman adımındaki bilgiyi saklayarak ve sonraki adımlarla bu bilgiyi güncelleyerek çalışır.
- **RNN Temel Özellikleri:**
    - Zaman boyutunda tekrar
    - Sekans verisi için uygun

![RNN Mimarisi](https://media.geeksforgeeks.org/wp-content/uploads/20260429164557969550/introduction_to_recurrent_neural_network.webp)
- **Zaman Serisi:** Zaman içerisinde ardışık olarak kaydedilen *(finans, iklim, sensör vs.)* verilerdir.
- **Sekans Verisi:** Doğal sıralı bağlılıkları olan *(doğal dil, konuşma tanıma, DNA dizileri vs.)* verilerdir.
- **Dil ve Zaman Serisi Verilerinde Dizisel Bağımlılık:** Zaman serisi ve dil verilerinde, her bir öğe sırasıyla önceki öğelere bağımlıdır *(Cümledeki kelimenin anlamının bir önceki kelimelere bağımlı olması, hava sıcaklığı tahmininde tahmin edilecek durumun daha önceki günlere bağımlı olması vs.)*.
- **Standart Sinir Ağları Sekans Verilerinde Neden Yetersizdir:**
    - Sabit girdi/çıktı
    - Zaman bağımlılığı
    - Geçmiş bilgiyi kaybetme
- **Vanishing Gradient Sorunu:**
    - RNN'lerde eğitim sırasında ortay çıkar.
    - Backpropagation sırasında, gradyanlar çok küçük hale gelir ve bu uzun süreli bağımlılıkların öğrenilmesini zorlaştırır.
    - Her zaman adımında alınan zincirleme türevler zaman içinde küçülür ve neredeyse sıfıra yaklaşır. Bu durumda önceki adımlardaki bilginin etkisi kaybolur. Bu sebeple, kısa dönem bağımlılıklar öğrenilirken uzun dönem bağımlılıkların kaybolur.
- **Kullanım Alanları:**
    - Dil modelleme
    - Makine çevirisi
    - Duygu analizi
    - Konuşma tanıma
    - Metin üretimi

In [1]:
import numpy as np
import pandas as pd

from gensim.models import Word2Vec
from keras.preprocessing.sequence import pad_sequences
from keras.models import Sequential
from keras.layers import SimpleRNN, Dense, Embedding
from keras.layers import TextVectorization
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [2]:
imdb_df = pd.read_csv("IMDB Dataset.csv")
imdb_df.head(5)

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [3]:
import re
from bs4 import BeautifulSoup as bs
from nltk.corpus import stopwords

stop_words_eng = stopwords.words("english")

def clean_text(text):
    text = text.lower()
    text = bs(text, "html.parser").get_text()

    # Text temizliği:
    # Kelimeler arasındaki '-' karakterleri
    # Lookarounds:   (?=) - positive lookahead
    #                (?!) - negative lookahead
    #                (?<=) - positive lookbehind
    #                (?<!) - negative lookbehind
    text = re.sub(r"(?<=\w)-(?=\w)", " ", text)
    # Harf olmayan karakterlerin tamamı:
    text = re.sub(r"[^A-Za-z\s]", "", text)
    # Peşpeşe birden fazla kez gelen boşluk karakterleri:
    text = re.sub(r"\s{2,}", "", text)
    # Sık kullanılan farklı yazımlar: \b => word boundary 
    text = re.sub(r"cannot", "can not", text)
    text = re.sub(r"\bim\b", "i am", text)

    ttokens = text.split()
    text = " ".join([word for word in ttokens if word not in stop_words_eng])

    return text

In [4]:
clean_text(imdb_df["review"].iloc[1])

'wonderful little production filming technique unassuming old time bbc fashion gives comforting sometimes discomforting sense realism entire piece actors extremely well chosen michael sheen got polari voices pat truly see seamless editing guided references williams diary entries well worth watching terrificly written performed piece masterful production one great masters comedy life realism really comes home little things fantasy guard rather use traditional dream techniques remains solid disappears plays knowledge senses particularly scenes concerning orton halliwell sets particularly flat halliwells murals decorating every surface terribly well done'

In [5]:
cleaned_reviews = imdb_df["review"].iloc[:5000].apply(clean_text)

In [15]:
vectorize_layer = TextVectorization(output_mode="int", max_tokens=5000, output_sequence_length=500)
vectorize_layer.adapt(cleaned_reviews, batch_size=500)

In [16]:
sequences = vectorize_layer(cleaned_reviews)

In [17]:
sequences.shape

TensorShape([5000, 500])

In [18]:
# label encoding
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(imdb_df["sentiment"].iloc[:5000])
y[:5]

array([1, 1, 1, 0, 1])

In [28]:
sequences.numpy()

array([[   4, 1718,  806, ...,    0,    0,    0],
       [ 250,   32,  240, ...,    0,    0,    0],
       [  94,  250,   24, ...,    0,    0,    0],
       ...,
       [   1,    1,  809, ...,    0,    0,    0],
       [1402,    2, 2417, ...,    0,    0,    0],
       [  82, 1408,   29, ...,    0,    0,    0]], shape=(5000, 500))

In [29]:
x_train, x_test, y_train, y_test = train_test_split(sequences.numpy(), y, test_size=0.33, random_state=60)

In [70]:
treviews_list = [review.split() for review in cleaned_reviews]

In [71]:
# Metin temsili:
word2vec_model = Word2Vec(treviews_list, window = 5, vector_size=50, min_count=10, sg = 0)

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


In [86]:
# Eğitim sırasında oluşuturulan W2V modelindeki kelime vektörleri embedding matrix'e ekledik.

embedding_dim = 50
embedding_matrix = np.zeros((len(vectorize_layer.get_vocabulary(include_special_tokens=False))+2, embedding_dim))
i = 0
for word in vectorize_layer.get_vocabulary(include_special_tokens=False):
    if word in word2vec_model.wv:
        embedding_matrix[i] = word2vec_model.wv[word]
    i+=1

In [90]:
model = Sequential()

model.add(Embedding(len(vectorize_layer.get_vocabulary()), output_dim = embedding_dim, weights = [embedding_matrix], input_length=5000, trainable=False))
model.add(SimpleRNN(50))
model.add(Dense(1, activation="sigmoid"))


model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

model.fit(x_train, y_train, epochs = 50, batch_size=500, validation_data=(x_test,y_test))

Epoch 1/50
7/7 ━━━━━━━━━━━━━━━━━━━━ 4s 378ms/step - accuracy: 0.5036 - loss: 0.7013 - val_accuracy: 0.5030 - val_loss: 0.6950
Epoch 2/50
7/7 ━━━━━━━━━━━━━━━━━━━━ 2s 304ms/step - accuracy: 0.5084 - loss: 0.6947 - val_accuracy: 0.5030 - val_loss: 0.6941
Epoch 3/50
7/7 ━━━━━━━━━━━━━━━━━━━━ 2s 306ms/step - accuracy: 0.4875 - loss: 0.6953 - val_accuracy: 0.5030 - val_loss: 0.6940
Epoch 4/50
7/7 ━━━━━━━━━━━━━━━━━━━━ 2s 309ms/step - accuracy: 0.5090 - loss: 0.6937 - val_accuracy: 0.5030 - val_loss: 0.6945
Epoch 5/50
7/7 ━━━━━━━━━━━━━━━━━━━━ 2s 298ms/step - accuracy: 0.4952 - loss: 0.6933 - val_accuracy: 0.4945 - val_loss: 0.6936
Epoch 6/50
7/7 ━━━━━━━━━━━━━━━━━━━━ 2s 277ms/step - accuracy: 0.5012 - loss: 0.6935 - val_accuracy: 0.5042 - val_loss: 0.6935
Epoch 7/50
7/7 ━━━━━━━━━━━━━━━━━━━━ 2s 316ms/step - accuracy: 0.5116 - loss: 0.6923 - val_accuracy: 0.4964 - val_loss: 0.6932
Epoch 8/50
7/7 ━━━━━━━━━━━━━━━━━━━━ 2s 316ms/step - accuracy: 0.5030 - loss: 0.6927 - val_accuracy: 0.5048 - val_loss:

In [89]:
test_loss, test_acc = model.evaluate(x_test, y_test)

print(f"Test loss: {test_loss}\nTest accuracy: {test_acc}")

52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.5048 - loss: 0.6928
Test loss: 0.6927557587623596
Test accuracy: 0.5048484802246094


In [91]:
test = "this movie was really bad awful even"
test_sequences = vectorize_layer(test)

pred = model.predict(test_sequences)

print(pred)

ValueError: Exception encountered when calling Sequential.call().

[1mInvalid input shape for input Tensor("sequential_12_1/Cast:0", shape=(32,), dtype=float32) with name 'keras_tensor_20' and path ''. Expected shape (None, 500), but input has incompatible shape (32,)[0m

Arguments received by Sequential.call():
  • inputs=tf.Tensor(shape=(32,), dtype=int64)
  • training=False
  • mask=None
  • kwargs=<class 'inspect._empty'>